In [29]:
import os, time, pandas as pd
from dataclasses import dataclass, asdict
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- config --------------------
SITE_ROOT = "https://expinterweb.mites.gob.es"
BASE_URL = f"{SITE_ROOT}/regcon/pub/buscadorTextosEstatal"

def make_driver(download_path, headless=False):
    opts = Options()
    if headless: opts.add_argument("--headless=new")
    
    # POWERFUL SETTINGS: Tells Chrome to auto-download PDFs
    prefs = {
        "download.default_directory": download_path,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True # Crucial: Don't open in browser, save to disk!
    }
    opts.add_experimental_option("prefs", prefs)
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

@dataclass
class AgreementSpain:
    codigo: str; denominacion: str; naturaleza: str; autoridad_laboral: str
    estado_vigencia: str; pdf_saved_as: str | None

# -------------------- scraper --------------------
def scrape_spain_browser_download(out_dir="spain_data", target_count=10):
    abs_out_dir = os.path.abspath(out_dir)
    pdf_dir = os.path.join(abs_out_dir, "pdfs")
    if not os.path.exists(pdf_dir): os.makedirs(pdf_dir, exist_ok=True)
    
    d = make_driver(pdf_dir, headless=False)
    results = []

    try:
        print(f"Loading {BASE_URL}...")
        d.get(BASE_URL)
        time.sleep(2)

        # Apply Search
        WebDriverWait(d, 10).until(EC.presence_of_element_located((By.ID, "texto"))).send_keys("Convenio")
        Select(d.find_element(By.ID, "idNaturaleza")).select_by_visible_text("CONVENIO COLECTIVO")
        d.find_element(By.ID, "_buscar").click()

        WebDriverWait(d, 20).until(EC.presence_of_element_located((By.XPATH, "//table//tr[td]")))
        
        # We process one by one
        for i in range(target_count):
            # Re-fetch rows to stay fresh
            rows = d.find_elements(By.XPATH, "//table//tr[td]")
            if i >= len(rows): break
            
            cols = rows[i].find_elements(By.TAG_NAME, "td")
            codigo = cols[0].text.strip()
            print(f"[{i+1}/{target_count}] Downloading: {codigo}...")

            ag = AgreementSpain(
                codigo=codigo,
                denominacion=cols[1].text.strip(),
                naturaleza=cols[2].text.strip(),
                autoridad_laboral=cols[3].text.strip(),
                estado_vigencia=cols[4].text.strip(),
                pdf_saved_as=os.path.join(pdf_dir, f"{codigo}.pdf")
            )

            # CLICK THE LINK
            try:
                ver_link = cols[-1].find_element(By.TAG_NAME, "a")
                # We use JS click to ensure the session is passed
                d.execute_script("arguments[0].click();", ver_link)
                
                # Wait for the file to actually land in the folder
                timeout = 15
                start_time = time.time()
                while time.time() - start_time < timeout:
                    # Note: We look for the file but ignore '.crdownload' temp files
                    files = os.listdir(pdf_dir)
                    if any(f.startswith(codigo) and not f.endswith(".crdownload") for f in files):
                        print(f"   ---> SUCCESS: File landed.")
                        break
                    time.sleep(1)
            except Exception as e:
                print(f"   ---> Error: {e}")

            results.append(ag)

    finally:
        d.quit()
        if results:
            df = pd.DataFrame([asdict(r) for r in results])
            csv_path = os.path.join(abs_out_dir, "spain_convenios.csv")
            df.to_csv(csv_path, index=False)
            print(f"\nFinal Metadata saved to: {csv_path}")
            return df

# RUN
scrape_spain_browser_download(target_count=10)

Loading https://expinterweb.mites.gob.es/regcon/pub/buscadorTextosEstatal...
[1/10] Downloading: 41101121012021...
[2/10] Downloading: 43002462012001...
[3/10] Downloading: 43002072012000...
[4/10] Downloading: 47000892011991...
[5/10] Downloading: 47000582011981...
[6/10] Downloading: 52100185012019...
[7/10] Downloading: 33100771012024...
[8/10] Downloading: 90002472011982...
[9/10] Downloading: 36002642011995...
[10/10] Downloading: 36104940012017...

Final Metadata saved to: /Users/lukeschreiber/Downloads/RA Work/Collective-Bargaining-Agreements-Web-Scraping/spain_data/spain_convenios.csv


,codigo,denominacion,naturaleza,autoridad_laboral,estado_vigencia,pdf_saved_as
0,41101121012021,SOCIEDAD DE FOMENTO AGRICOLA CASTELLONENSE S.A.,CONVENIO COLECTIVO,Sevilla,,/Users/lukeschreiber/Downloads/RA Work/Collect...
1,43002462012001,"ASOCIACION NUCLEAR ASCO-VANDELLOS II, AIE",CONVENIO COLECTIVO,Tarragona,,/Users/lukeschreiber/Downloads/RA Work/Collect...
2,43002072012000,Consorci d'Aigües de Tarragona,CONVENIO COLECTIVO,Tarragona,,/Users/lukeschreiber/Downloads/RA Work/Collect...
3,47000892011991,"ARROYO, S.A.",CONVENIO COLECTIVO,Valladolid,,/Users/lukeschreiber/Downloads/RA Work/Collect...
4,47000582011981,VALLADOLID AUTOMOVIL S.A. (VASA),CONVENIO COLECTIVO,Valladolid,,/Users/lukeschreiber/Downloads/RA Work/Collect...
5,52100185012019,CONVENIO COLECTIVO SECTORIAL PARA EL TRANSPORT...,CONVENIO COLECTIVO,Melilla,,/Users/lukeschreiber/Downloads/RA Work/Collect...
6,33100771012024,ELECTRONIC TRAFIC S.A.,CONVENIO COLECTIVO,Asturias,,/Users/lukeschreiber/Downloads/RA Work/Collect...
7,90002472011982,"HERMANDAD FARMACEUTICA DEL MEDITERRANEO, SOCIE...",CONVENIO COLECTIVO,Estatal,,/Users/lukeschreiber/Downloads/RA Work/Collect...
8,36002642011995,TRATAMIENTO INDUSTRIAL DE AGUAS SA (TRAINASA),CONVENIO COLECTIVO,Pontevedra,,/Users/lukeschreiber/Downloads/RA Work/Collect...
9,36104940012017,"DALPHIMETAL ESPAÑA, S.L.U. - CENTRO DE TRABAJO...",CONVENIO COLECTIVO,Pontevedra,,/Users/lukeschreiber/Downloads/RA Work/Collect...


In [30]:
# import time
# from selenium.webdriver.common.by import By

# d = make_driver(headless=False)
# try:
#     print("Loading main page...")
#     d.get("https://expinterweb.mites.gob.es/regcon/pub/consultaPublicaEstatal")
    
#     print("\n>>> QUICK! Please click 'BUSCAR EN TEXTOS' on the left menu in the Chrome window! <<<")
#     print("You have 15 seconds to click it...")
#     time.sleep(15) 

#     print("\n--- NEW PAGE URL ---")
#     print(d.current_url)

#     print("\n--- ALL TEXT BOXES ---")
#     inputs = d.find_elements(By.XPATH, "//input[@type='text']")
#     for i in inputs:
#         print(f"ID: {i.get_attribute('id')} | Name: {i.get_attribute('name')}")

#     print("\n--- ALL DROPDOWNS (SELECT) ---")
#     selects = d.find_elements(By.TAG_NAME, "select")
#     for s in selects:
#         print(f"ID: {s.get_attribute('id')} | Name: {s.get_attribute('name')}")

#     print("\n--- ALL BUTTONS ---")
#     buttons = d.find_elements(By.XPATH, "//button | //input[@type='submit'] | //input[@type='button']")
#     for b in buttons:
#         text = b.text.strip() if b.text.strip() else b.get_attribute('value')
#         print(f"Tag: {b.tag_name} | ID: {b.get_attribute('id')} | Text/Value: {text}")

# finally:
#     d.quit()